# Bronze → Silver: Account Domain — Account.txt
## silver.account (MERGE on CA_ID, latest wins)

**Sources:**
- `bronze.customermgmt` → Account actions from CustomerMgmt.xml (B1: NEW, ADDACCT, UPDACCT, CLOSEACCT)
- `bronze.account` → CDC updates from Account.txt (B2/B3: I=Insert, U=Update, D=Delete excluded)

**Processing Pattern:** CDC with FLAG → MERGE on natural key (latest wins)

**Silver Layer Contract (per MD):**
- DROPPED: `_ingest_ts` (bronze-only), `_source_file` (no longer needed)
- ADDED: `_load_ts` (silver load timestamp)
- CARRIED: `_batch`, `_run_id`
- Type cast: CA_ID→BIGINT, CA_C_ID→BIGINT, CA_B_ID→BIGINT, CA_TAX_ST→INT
- Enrichment: Join `silver.statustype` for StatusDesc
- Timestamp resolution: Join `silver.batchdate` for CDC records

**Expected Counts:**
| After Batch | Count |
|:--|:--|
| B1 (XML only) | 30,470 |
| B2 (+70 I, +30 U) | 30,540 |
| B3 (+70 I, +30 U) | 30,610 |

###### Author: Prajwol Regmi

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import (
    current_timestamp, col, row_number, lit, when, coalesce,
    to_timestamp, to_date, concat_ws, count, sum as _sum, md5
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType, LongType, DecimalType, TimestampType, StringType
)
from pyspark.sql import Row
from delta.tables import DeltaTable
from datetime import datetime

# ─── CONFIG ──────────────────────────────────────────────────────────────
team_name = "team_lemma"
catalog_name = f"charles_schwab_retailbrokerage_dev_{team_name}"
bronze_schema = "bronze"
silver_schema = "silver"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {silver_schema}")

# Target table fully qualified name
TARGET_TABLE = f"{catalog_name}.{silver_schema}.account"

print(f"Catalog: {catalog_name}")
print(f"Source: {bronze_schema} | Target: {silver_schema}")
print(f"Target Table: {TARGET_TABLE}")
print(f"Mode: MERGE on CA_ID (latest wins, CDC priority > XML)")

In [0]:
# ============================================================
# SOURCE 1: CustomerMgmt.xml Account Actions (Batch 1)
# ============================================================
# From MD Section 6A:
#   - Parse XML to extract <Account> nodes nested inside <Customer>
#   - Handle ActionTypes that affect accounts:
#     - NEW: New customer + initial account (create record)
#     - ADDACCT: Additional account for existing customer
#     - UPDACCT: Update account (triggers SCD-2 version in Gold)
#     - CLOSEACCT: Close account (Status = INAC)
#   - Extract: CA_ID, CA_B_ID, CA_C_ID, CA_NAME, CA_TAX_ST, CA_ST_ID
#   - Link to parent Customer via C_ID
# ============================================================

print("="*70)
print("SILVER.ACCOUNT: Bronze → Silver Processing")
print("="*70)
print("\n[Source 1] CustomerMgmt.xml Account Actions (Batch 1)")
print("-"*70)

# Read from bronze.customermgmt
# Filter: ActionType that affects accounts AND CA_ID is populated
xml_raw = spark.table(f"{catalog_name}.{bronze_schema}.customermgmt")

xml_accounts = (
    xml_raw
    .filter(
        (col("ActionType").isin("NEW", "ADDACCT", "UPDACCT", "CLOSEACCT")) &
        (col("CA_ID").isNotNull()) & (col("CA_ID") != "")
    )
    .select(
        col("CA_ID"),
        col("C_ID").alias("CA_C_ID"),            # Customer ID from parent XML node
        col("CA_B_ID"),                           # Broker ID
        col("CA_NAME"),                           # Account name
        col("CA_TAX_ST"),                         # Tax status (0/1/2)
        # Derive CA_ST_ID from ActionType:
        #   CLOSEACCT → 'INAC' (account closed)
        #   All others → 'ACTV' (account active)
        when(col("ActionType") == "CLOSEACCT", lit("INAC"))
            .otherwise(lit("ACTV")).alias("CA_ST_ID"),
        # ActionTS is the authoritative business timestamp
        to_timestamp(col("ActionTS")).alias("update_ts"),
        col("ActionType"),                        # Keep for diagnostics
        col("_batch_id"),
        col("_run_id"),
        col("_ingest_ts")
    )
)

xml_count = xml_accounts.count()

# Distribution analysis
print(f"  Total XML account actions: {xml_count}")
xml_accounts.groupBy("ActionType").count().orderBy("ActionType").show(truncate=False)

# Distinct CA_IDs from XML (should be 30,470)
xml_distinct = xml_accounts.select("CA_ID").distinct().count()
print(f"  Distinct CA_IDs from XML: {xml_distinct} (expected: 30,470)")

In [0]:
# ============================================================
# SOURCE 2: Account.txt CDC (Batch 2, Batch 3)
# ============================================================
# From MD:
#   - CDC_FLAG: I=Insert (70/batch), U=Update (30/batch), D=Delete (excluded)
#   - CDC_DSN: Data Sequence Number (ordering within batch)
#   - B2/B3 only (no Account.txt in Batch 1)
#   - Join silver.batchdate for timestamp resolution:
#     B2 → 2017-07-08, B3 → 2017-07-09
#   - These dates are AFTER all XML actions (≤2017-07-07)
#     ensuring CDC priority > XML for same CA_ID
# ============================================================

print("\n[Source 2] Account.txt CDC (Batch 2/3)")
print("-"*70)

# Read silver.batchdate for timestamp resolution
# Maps _batch_id → actual business date
batchdate_df = (
    spark.table(f"{catalog_name}.{silver_schema}.batchdate")
    .select(
        col("_batch_id"),
        to_timestamp(col("batchdate")).alias("batch_ts")
    )
)

print("  Batch date mapping:")
batchdate_df.show(truncate=False)

# Read bronze.account (B2/B3 CDC)
bronze_account = spark.table(f"{catalog_name}.{bronze_schema}.account")

print(f"  Bronze.account total rows: {bronze_account.count()}")
print(f"  CDC_FLAG distribution:")
bronze_account.groupBy("_batch_id", "CDC_FLAG").count().orderBy("_batch_id", "CDC_FLAG").show()

# Filter: Exclude CDC_FLAG='D' (delete/close records)
# Per MD: "D records excluded" from silver.account
# Note: No D records exist in test data, but defensive coding
cdc_accounts = (
    bronze_account
    .filter(col("CDC_FLAG") != "D")
    .join(batchdate_df, on="_batch_id", how="left")
    .select(
        col("CA_ID"),
        col("CA_C_ID"),
        col("CA_B_ID"),
        col("CA_NAME"),
        col("CA_TAX_ST"),
        col("CA_ST_ID"),
        # Use batchdate as the business timestamp for CDC records
        # This ensures CDC records (B2=2017-07-08, B3=2017-07-09)
        # naturally win over XML records (≤2017-07-07) in dedup
        col("batch_ts").alias("update_ts"),
        lit(None).cast(StringType()).alias("ActionType"),  # No ActionType for CDC
        col("_batch_id"),
        col("_run_id"),
        col("_ingest_ts")
    )
)

cdc_count = cdc_accounts.count()
print(f"  CDC accounts after D-exclusion: {cdc_count}")

# Distinct new CA_IDs from CDC (should be 140 total new = 70+70)
cdc_new = bronze_account.filter(col("CDC_FLAG") == "I").select("CA_ID").distinct().count()
print(f"  Distinct NEW CA_IDs from CDC (I flag): {cdc_new} (expected: 140)")

In [0]:
# ============================================================
# UNION + DEDUP + TYPE CAST + STATUS ENRICHMENT
# ============================================================
# From MD:
#   - UNION CustomerMgmt.xml Account nodes + Account.txt CDC
#   - Order by ActionTS/CDC timestamp (latest wins)
#   - CDC_FLAG exists but "Silver uses latest-wins dedup,
#     not explicit I/U/D routing"
#   - Dedup by CA_ID (ROW_NUMBER PARTITION BY CA_ID
#     ORDER BY update_ts DESC, _ingest_ts DESC)
#   - Enrich: Join silver.statustype for StatusDesc (ST_NAME)
# ============================================================

print("\n[Transform] Union + Dedup + Type Cast + Enrich")
print("-"*70)

# ── STEP 1: UNION both sources with common schema ───────────────────
union_accounts = xml_accounts.unionByName(cdc_accounts)
total_source_count = union_accounts.count()
print(f"  1. UNION combined rows: {total_source_count} (XML={xml_count} + CDC={cdc_count})")

# ── STEP 2: DEDUPLICATE by CA_ID (latest update_ts wins) ─────────────
# ROW_NUMBER() PARTITION BY CA_ID ORDER BY update_ts DESC, _ingest_ts DESC
# Tiebreaker: _ingest_ts DESC ensures most recent ingestion wins
# CDC records (B2/B3) have dates 2017-07-08/09 which are AFTER
# all XML records (≤2017-07-07) → CDC priority > XML guaranteed
dedup_window = Window.partitionBy("CA_ID").orderBy(
    col("update_ts").desc(),
    col("_ingest_ts").desc()
)

deduped_accounts = (
    union_accounts
    .withColumn("_row_num", row_number().over(dedup_window))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

dedup_count = deduped_accounts.count()
print(f"  2. After dedup (distinct CA_IDs): {dedup_count} (expected: 30,610)")

# ── STEP 3: TYPE CAST business columns ─────────────────────────────
# Per MD Silver Layer Contract:
#   CA_ID → BIGINT, CA_C_ID → BIGINT, CA_B_ID → BIGINT
#   CA_TAX_ST → INT, CA_NAME → STRING (no cast)
#   CA_ST_ID → STRING (status code: ACTV, INAC)
typed_accounts = deduped_accounts.select(
    col("CA_ID").cast(LongType()).alias("CA_ID"),
    col("CA_C_ID").cast(LongType()).alias("CA_C_ID"),
    col("CA_B_ID").cast(LongType()).alias("CA_B_ID"),
    col("CA_NAME"),                                         # STRING
    col("CA_TAX_ST").cast(IntegerType()).alias("CA_TAX_ST"),
    col("CA_ST_ID"),                                        # STRING (ACTV/INAC)
    col("update_ts"),                                       # Keep for lineage
    col("_batch_id"),
    col("_run_id")
)

print(f"  3. Type cast applied (CA_ID→BIGINT, CA_C_ID→BIGINT, CA_B_ID→BIGINT, CA_TAX_ST→INT)")

# ── STEP 4: ENRICH with StatusDesc from silver.statustype ────────────
# Per MD: "StatusDesc (enriched from statustype)"
# Join silver.statustype on CA_ST_ID = ST_ID to get ST_NAME
statustype_df = (
    spark.table(f"{catalog_name}.{silver_schema}.statustype")
    .select(
        col("ST_ID"),
        col("ST_NAME").alias("CA_ST_DESC")  # Status description
    )
)

enriched_accounts = (
    typed_accounts
    .join(statustype_df, typed_accounts["CA_ST_ID"] == statustype_df["ST_ID"], "left")
    .drop("ST_ID")
)

print(f"  4. Status enrichment applied (CA_ST_DESC from silver.statustype)")

# ── STEP 5: Apply Silver audit columns ─────────────────────────────
# Per MD Silver Layer Contract:
#   DROPPED: _ingest_ts, _source_file
#   ADDED: _load_ts
#   CARRIED: _batch, _run_id
silver_account_df = (
    enriched_accounts
    .select(
        # Business columns
        "CA_ID", "CA_C_ID", "CA_B_ID", "CA_NAME",
        "CA_TAX_ST", "CA_ST_ID", "CA_ST_DESC",
        # Audit columns (per MD spec)
        col("_batch_id").alias("_batch"),           # CARRIED
        col("_run_id"),                             # CARRIED
    )
    .withColumn("_load_ts", current_timestamp())    # ADDED
)

print(f"  5. Silver audit columns applied (drop _ingest_ts/_source_file, add _load_ts)")
print(f"\n  Final silver.account DataFrame: {silver_account_df.count()} rows")
print(f"  Schema:")
for name, dtype in silver_account_df.dtypes:
    print(f"    {name:20s} {dtype}")

In [0]:
# ============================================================
# WRITE: MERGE INTO silver.account
# ============================================================
# Per MD: Mode = MERGE for tables with incremental updates
# Pattern: CDC with FLAG → MERGE on natural key (latest wins)
#
# Behavior:
#   - First run (table doesn't exist): CREATE via overwrite
#   - Subsequent runs: MERGE upsert on CA_ID
#     - MATCHED: UPDATE all columns (account changed)
#     - NOT MATCHED: INSERT (new account)
# Idempotent: re-running produces identical result
# ============================================================

print("\n[Write] MERGE into silver.account")
print("-"*70)

# Check if target table exists
table_exists = spark.catalog.tableExists(TARGET_TABLE)

if not table_exists:
    # First run: Create the table with full dataset
    print(f"  Table does not exist. Creating: {TARGET_TABLE}")
    silver_account_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TARGET_TABLE)
    write_mode = "CREATE"
else:
    # Subsequent runs: MERGE (upsert on CA_ID)
    print(f"  Table exists. Executing MERGE: {TARGET_TABLE}")
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)
    
    delta_target.alias("target").merge(
        silver_account_df.alias("source"),
        "target.CA_ID = source.CA_ID"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    write_mode = "MERGE"

# ── Validate target count ────────────────────────────────────────
account_target_count = spark.table(TARGET_TABLE).count()
account_run_id = silver_account_df.select("_run_id").first()[0]

expected_count = 30610
match_status = "✓ PASS" if account_target_count == expected_count else "✗ MISMATCH"

print(f"\n  Write mode: {write_mode}")
print(f"  Target count: {account_target_count} (expected: {expected_count}) [{match_status}]")
print(f"  Carried run_id: {account_run_id}")

In [0]:
# ============================================================
# DATA QUALITY VALIDATION
# ============================================================
# Verify silver.account meets all quality expectations:
#   1. Row count matches expected (30,610)
#   2. Primary key (CA_ID) is unique and not null
#   3. Foreign keys are valid (CA_C_ID, CA_B_ID not null)
#   4. Status codes are valid (ACTV, INAC only)
#   5. CA_TAX_ST in valid range (0, 1, 2)
#   6. No orphan status codes (all enriched successfully)
# ============================================================

print("\n[DQ] Data Quality Validation")
print("-"*70)

silver_account = spark.table(TARGET_TABLE)
dq_results = []

# DQ-1: Row count validation
actual_count = silver_account.count()
dq_1_pass = actual_count == 30610
dq_results.append({"rule": "ROW_COUNT", "expected": "30610", "actual": str(actual_count), "status": "PASS" if dq_1_pass else "FAIL"})
print(f"  DQ-1 Row Count: {actual_count} (expected 30,610) [{'PASS' if dq_1_pass else 'FAIL'}]")

# DQ-2: Primary Key uniqueness (CA_ID)
pk_dupes = silver_account.groupBy("CA_ID").count().filter(col("count") > 1).count()
dq_2_pass = pk_dupes == 0
dq_results.append({"rule": "PK_UNIQUE_CA_ID", "expected": "0 dupes", "actual": str(pk_dupes), "status": "PASS" if dq_2_pass else "FAIL"})
print(f"  DQ-2 PK Uniqueness (CA_ID): {pk_dupes} duplicates [{'PASS' if dq_2_pass else 'FAIL'}]")

# DQ-3: CA_ID not null
null_ca_id = silver_account.filter(col("CA_ID").isNull()).count()
dq_3_pass = null_ca_id == 0
dq_results.append({"rule": "NOT_NULL_CA_ID", "expected": "0 nulls", "actual": str(null_ca_id), "status": "PASS" if dq_3_pass else "FAIL"})
print(f"  DQ-3 CA_ID Not Null: {null_ca_id} nulls [{'PASS' if dq_3_pass else 'FAIL'}]")

# DQ-4: Valid status codes (CA_ST_ID in ACTV, INAC)
invalid_status = silver_account.filter(~col("CA_ST_ID").isin("ACTV", "INAC")).count()
dq_4_pass = invalid_status == 0
dq_results.append({"rule": "VALID_STATUS_CODES", "expected": "0 invalid", "actual": str(invalid_status), "status": "PASS" if dq_4_pass else "FAIL"})
print(f"  DQ-4 Valid Status Codes: {invalid_status} invalid [{'PASS' if dq_4_pass else 'FAIL'}]")

# DQ-5: CA_TAX_ST in valid range (0, 1, 2) - allow nulls (CLOSEACCT may have null)
invalid_tax = silver_account.filter(
    col("CA_TAX_ST").isNotNull() & ~col("CA_TAX_ST").isin(0, 1, 2)
).count()
dq_5_pass = invalid_tax == 0
dq_results.append({"rule": "VALID_TAX_STATUS", "expected": "0 invalid", "actual": str(invalid_tax), "status": "PASS" if dq_5_pass else "FAIL"})
print(f"  DQ-5 Valid Tax Status (0/1/2): {invalid_tax} invalid [{'PASS' if dq_5_pass else 'FAIL'}]")

# DQ-6: Status enrichment success (CA_ST_DESC not null where CA_ST_ID exists)
# Only run if CA_ST_DESC column exists in table (depends on whether table was
# rebuilt with enrichment step or pre-existing from older schema)
if "CA_ST_DESC" in silver_account.columns:
    null_desc = silver_account.filter(
        col("CA_ST_ID").isNotNull() & col("CA_ST_DESC").isNull()
    ).count()
    dq_6_pass = null_desc == 0
    dq_results.append({"rule": "STATUS_ENRICHMENT", "expected": "0 failed", "actual": str(null_desc), "status": "PASS" if dq_6_pass else "FAIL"})
    print(f"  DQ-6 Status Enrichment: {null_desc} unenriched [{'PASS' if dq_6_pass else 'FAIL'}]")
else:
    dq_results.append({"rule": "STATUS_ENRICHMENT", "expected": "column present", "actual": "CA_ST_DESC missing", "status": "SKIP"})
    print(f"  DQ-6 Status Enrichment: SKIPPED (CA_ST_DESC column not in table — drop & recreate table to add enrichment)")

# DQ-7: Audit column completeness (_run_id, _batch, _load_ts not null)
null_audit = silver_account.filter(
    col("_run_id").isNull() | col("_batch").isNull() | col("_load_ts").isNull()
).count()
dq_7_pass = null_audit == 0
dq_results.append({"rule": "AUDIT_COMPLETENESS", "expected": "0 nulls", "actual": str(null_audit), "status": "PASS" if dq_7_pass else "FAIL"})
print(f"  DQ-7 Audit Completeness: {null_audit} rows missing audit cols [{'PASS' if dq_7_pass else 'FAIL'}]")

# Summary
total_pass = sum(1 for r in dq_results if r["status"] == "PASS")
total_rules = len(dq_results)
print(f"\n  DQ Summary: {total_pass}/{total_rules} rules passed")

# Log DQ results to operations layer
for dq in dq_results:
    if dq["status"] == "SKIP":
        continue
    failed_rows = 0 if dq["status"] == "PASS" else int(dq["actual"]) if dq["actual"].isdigit() else 1
    log_dq_result(
        spark=spark,
        run_id=account_run_id,
        table_name="silver.account",
        rule_name=dq["rule"],
        failed_rows=failed_rows,
        total_rows=actual_count
    )

In [0]:
# ============================================================
# OPERATIONS LOGGING: Pipeline Reconciliation + Audit Event
# ============================================================
# Per project standard (from 02_common_utils/operations):
#   1. log_pipeline_recon() - source vs target count comparison
#   2. log_audit_event() - write operation audit trail
# ============================================================

print("\n[Ops] Operations Logging")
print("-"*70)

# 1. Pipeline Reconciliation
log_pipeline_recon(
    spark=spark,
    run_id=account_run_id,
    batch_id="Batch3",               # Final batch processed (cumulative)
    domain="ACCOUNT",
    table_name="account",
    source_layer="bronze",
    target_layer="silver",
    source_count=total_source_count,  # Combined XML + CDC source rows
    target_count=account_target_count # Final deduped silver count
)
print(f"  ✓ log_pipeline_recon: source={total_source_count} → target={account_target_count}")

# 2. Audit Event
log_audit_event(
    spark=spark,
    run_id=account_run_id,
    batch="Batch3",
    layer="silver",
    table_name="account",
    operation=write_mode,             # CREATE or MERGE
    rows_affected=account_target_count
)
print(f"  ✓ log_audit_event: operation={write_mode}, rows={account_target_count}")

# 3. Domain Run Status
log_domain_run_status(
    spark=spark,
    run_id=account_run_id,
    batch="Batch3",
    domain_name="ACCOUNT_SILVER_ACCOUNT",
    status="COMPLETED"
)
print(f"  ✓ log_domain_run_status: ACCOUNT_SILVER_ACCOUNT = COMPLETED")

print(f"\n  All operations logged successfully (run_id: {account_run_id})")

In [0]:
# ============================================================
# VERIFICATION: silver.account
# ============================================================

print("[Verify] silver.account")
print("="*70)

silver_acct = spark.table(TARGET_TABLE)

# Count
print(f"  Total rows: {silver_acct.count()} (expected: 30,610)")

# Schema
print(f"\n  Schema:")
for c in silver_acct.dtypes:
    print(f"    {c[0]:20s} {c[1]}")

# Status distribution
print(f"\n  Status distribution:")
if "CA_ST_DESC" in silver_acct.columns:
    silver_acct.groupBy("CA_ST_ID", "CA_ST_DESC").count().orderBy("CA_ST_ID").show(truncate=False)
else:
    silver_acct.groupBy("CA_ST_ID").count().orderBy("CA_ST_ID").show(truncate=False)

# Tax status distribution
print(f"  Tax status distribution:")
silver_acct.groupBy("CA_TAX_ST").count().orderBy("CA_TAX_ST").show()

# Batch distribution (which batch provided the winning record)
print(f"  Winning record batch distribution:")
silver_acct.groupBy("_batch").count().orderBy("_batch").show()

# Sample data
print(f"  Sample records (first 15):")
display(silver_acct.orderBy("CA_ID").limit(15))